# kitchen: GPU tests on Colab

Runs the GPU stage of kitchen's build (stage D of `docs/plans/0.0.9-frame.md`): kitchen-compute on
**CUDA**, on Colab's free T4. Everything else (stages A–C) runs in CI or a cloud session; this is
only what needs a GPU.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
A first run takes about 15–25 minutes (Rust and the crates compile from scratch); keep the tab open.

## 1. The GPU

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

## 2. What to test

The branch of kitchen, and the packages and features to run. Change them to test a pull request's branch.

In [ ]:
REPO = "https://github.com/neuro-kitchen/kitchen"
BRANCH = "main"
TESTS = [
    # (package, features)
    ("kitchen-compute", "cuda"),
]

## 3. Rust and the build settings

Installs Rust for this session, and builds without debug information to keep the disk small.

In [ ]:
import os, subprocess
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    subprocess.run("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal", shell=True, check=True)
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
os.environ.update({
    "CARGO_INCREMENTAL": "0",
    "CARGO_PROFILE_DEV_DEBUG": "0",
    "CARGO_PROFILE_TEST_DEBUG": "0",
    "CARGO_BUILD_JOBS": str(os.cpu_count() or 2),
})
!rustc --version && cargo --version
!apt-get -qq install -y pkg-config libssl-dev > /dev/null

## 4. The code

In [ ]:
import os
if os.path.exists("kitchen"):
    !cd kitchen && git fetch -q origin {BRANCH} && git checkout -q FETCH_HEAD
else:
    !git clone -q --depth 1 -b {BRANCH} {REPO} kitchen
!cd kitchen && git log -1 --format='%h %s'

## 5. The tests

Each package with its GPU feature. The summary at the end says passed or failed for each.

In [ ]:
import subprocess
results = []
for package, features in TESTS:
    print(f"== {package} --features {features}", flush=True)
    r = subprocess.run(["cargo", "test", "-p", package, "--features", features], cwd="kitchen")
    results.append((package, features, "passed" if r.returncode == 0 else "FAILED"))
print()
for package, features, status in results:
    print(f"{package:20} {features:10} {status}")

## Notes

- **Saving time between sessions:** a Colab session starts clean. To keep the build, mount Google
  Drive (*Files → Mount Drive*) and set `os.environ["CARGO_TARGET_DIR"] = "/content/drive/MyDrive/kitchen-target"`
  before step 5; later runs then compile only what changed.
- **wgpu** needs Vulkan, which Colab's GPUs usually do not expose; test it on a machine with a display
  driver.
- dsp-kitchen and lucem: add their GPU packages to `TESTS` with their repository in step 2 once
  they move to kitchen 0.0.9.